In [ ]:
%pip install pypdf
%pip install -U langchain langchain-text-splitters langchain-ollama langchain-unstructured langchain-community unstructured
%pip install faiss-cpu
%pip install unstructured[pdf]

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull qwen2.5:7b

In [ ]:
#Large language models (like llama3.1) and embedding models (like nomic-embed-text) must be downloaded to your machine separately before your Python scripts can access them
!ollama pull nomic-embed-text

In [5]:
import os
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
# Modern Ollama Imports
from langchain_ollama import ChatOllama, OllamaEmbeddings

C:\Users\Usuario\AppData\Local\Temp\ipykernel_7468\1585371015.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# --- 1. Load Documents ---
def load_documents_from_directory(dir_path: str):
    # Find all files in the directory (e.g., PDFs)
    file_paths = glob.glob(f"{dir_path}/*.pdf*")

    if not file_paths:
        print(f"No files found in {dir_path}")
        return []

    print(f"Loading following files: {file_paths}")

    documents = []
    for file_path in file_paths:
        documents.extend(PyPDFLoader(file_path).load())
    return documents

# --- 2. Build or Load Vector Database ---
def create_vector_store(documents, embedding_model_name="nomic-embed-text"):
    # Initialize the modern Ollama embedding class
    embeddings = OllamaEmbeddings(model=embedding_model_name)

    # Split text into manageable chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)

    # Generate embeddings and populate FAISS database
    print(f"Generating embeddings for {len(chunks)} text chunks...")

    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

def format_docs(docs):
    formatted_docs = "\n\n".join(doc.page_content for doc in docs)

    print(f"\n------Retrieve docs:")
    for doc in docs:
      print(doc.page_content)
      print("\n")
    print("\n To be used as context for the RAG query.\n")

    return formatted_docs

# --- 3. Execute RAG Query Loop ---
def run_rag_pipeline(vector_store, model_name="qwen2.5:7b"):
    # Instantiate the modern ChatOllama class for generation
    llm = ChatOllama(model=model_name, temperature=0)

    # Define a retriever from our FAISS index
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    while True:
        print("\nAsk a question about your documents (or type 'exit'): ")
        query = input("Question: ")
        if query.lower() == 'exit':
            break

        # Retrieve relevant text context
        # relevant_docs = retriever.invoke(query)
        # format_docs = "\n\n".join([doc.page_content for doc in relevant_docs])

        #Ajusta el prompt para que el modelo genere respuestas basadas en el contexto recuperado
        prompt_instance = ChatPromptTemplate.from_template("""
            You are an expert assistant in aircraft accidents and aviation safety.
            Respond ONLY using information from the provided context.
            If the answer is not in the context, say: "No existe esa información en el documento."
            Do not invent data, dates, or figures, do not make things up. Always respond in Spanish.

            Context:
            {context}

            Question: {question}
            Answer:
            """)

        rag_chain = (
                {
                    "context": itemgetter("question") | retriever | format_docs,
                    "question": itemgetter("question")
                }
                | prompt_instance
                | llm
                | StrOutputParser()
        )

        # Generate and stream the response from Ollama
        print("\n--- Answer ---")
        for chunk in rag_chain.stream({"question": query}):
            print(chunk, end="", flush=True)
        print("\n--------------")

In [ ]:
# --- Main Execution ---
if __name__ == "__main__":
    DOCS_DIR = "rag_pdf_files"

    # 1. Ingest
    raw_docs = load_documents_from_directory(DOCS_DIR)

    if raw_docs:
        # 2. Embed & Index (Make sure to run: 'ollama pull nomic-embed-text' first)
        db = create_vector_store(raw_docs, embedding_model_name="nomic-embed-text")
        print("Index ready. You can now ask questions.")

        # 3. Query (Make sure to run: 'ollama pull qwen2.5:7b' first)
        run_rag_pipeline(db, model_name="qwen2.5:7b")
    else:
        print("No documents found to process.")